<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/neuronpedia_example/circuit_tracer_w_neuronpedia_example.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

## Using the Interpretune Neuronpedia Extension

This tutorial demonstrates Interpretune's integration with Circuit Tracer and Neuronpedia using Gemma-3-1B-IT, guiding the user through generating and sharing a basic model attribution graph.
Future tutorials will demonstrate the use of Interpretune's analytical abstractions to
generate, transform and share more advanced model attribution graphs in the context of world model analysis, research or AI agent/application interpretability maximization.


#### Notebook Parameters

This cell contains parameters that can be modified for different test configurations using papermill.


In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
use_baseline_salient_logits = True  # logits computation mode: True->salient logits, False->specific logits
use_baseline_transcoder_arch = True  # transcoder architecture: True->SingleLayerTranscoder, False->CrossLayerTranscoder
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)

In [3]:
# Core imports
import interpretune as it  # registered analysis ops will be available as it.<op> when analysis is imported

In [4]:
# this example assumes the user has either installed `interpretune` with the `examples` extra
# (or that circuit_tracer and neuronpedia are installed separately in addition to base interpretune dependencies)

# Import circuit tracer and required modules
from transformer_lens import ActivationCache  # noqa: F401
from datetime import datetime

from it_examples import _ACTIVE_PATCHES  # noqa: F401  # TODO: add note about this unless patched in SL before release
from it_examples.example_module_registry import MODULE_EXAMPLE_REGISTRY  # TODO: move to hub once implemented
from it_examples.utils.example_helpers import required_os_env
from it_examples.example_prompt_configs import GemmaPromptConfig
from interpretune import ITSessionConfig, ITSession
from interpretune.base.call import it_init

In [5]:
import os

env_path: str | None = None  # set to '/full/path/to/.env' to override

# Note: We need to set your API keys via environment variables:
# export NEURONPEDIA_API_KEY="your-production-key"
# export DEV_NEURONPEDIA_API_KEY="your-dev-key"  # if using dev API
use_localhost = os.environ.get("USE_LOCALHOST", "false").lower() == "true"
# Note there is a planned request for neuronpedia to qualify USE_LOCALHOST as NP_USE_LOCALHOST
os_env_reqs = [("DEV_NEURONPEDIA_API_KEY", "non-empty")] if use_localhost else [("NEURONPEDIA_API_KEY", "non-empty")]
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

In [6]:
# Load our demo config (this will be done from the hub once that is available)
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = MODULE_EXAMPLE_REGISTRY.get("rte_demo.gemma3.circuit_tracer.neuronpedia")
# Optionally override base_it_cfg.core_log_dir with the notebook parameter if provided
if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir
# configure our session with our desired adapter composition, core and circuit_tracer in this case
session_cfg = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)

# start our session
it_session = ITSession(session_cfg)

/home/speediedan/repos/interpretune/src/interpretune/config/shared.py:222: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'circuit_tracer_cfg': CircuitTracerConfig(backend='nnsight', model_name=None, transcoder_set='mwhanna/gemma-scope-2-1b-it/transcoder_all/width_16k_l0_small_affine', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=256, max_feature_nodes=8192, offload='cpu', lazy_encoder=None, lazy_decoder=True, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=None, analysis_target_tokens=['▁Dallas', '▁Austin'], target_token_ids=None, use_neuronpedia=True, intervention_scale_factor=1.0, intervention_max_influence_norm_scale=False, intervention_sign_aware_scale=True, intervention_value=None, intervention_value_source='top_feature_scores', intervention_constrained_layers=None, intervention_freeze_attention=None, interventio

[INFO] interpretune.extensions.neuronpedia: Neuronpedia package available


[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma3TextConfig' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma3TextConfig' object has no attribute '_pre_quantization_dtype'


In [7]:
# manual init for now
it_init(**it_session)
print("\nIT Session initialized successfully!")

[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NNSightReplacementModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NNSightReplacementModel.forward`, you can safely ignore this message.


Map:   0%|          | 0/277 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NNSightReplacementModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NNSightReplacementModel.forward`, you can safely ignore this message.


...

[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule


[INFO] interpretune.utils.logging: Input gradient requirements handled by circuit tracer internally.



IT Session initialized successfully!


### Example: Using Circuit Tracer to generate a custom graph and upload to Neuronpedia

In [8]:
# Generate and upload graph in one step
raw_prompt = "The capital of the state containing Dallas is"
ct_module = it_session.module

# instruction-tuned replacement models assert chat-formatted inputs
# (circuit-tracer's ensure_tokenized requires a <bos><start_of_turn>user prefix)
prompt = GemmaPromptConfig().apply_chat_template_fn(
    it_session.datamodule.tokenizer, raw_prompt, tokenize=False, add_generation_prompt=True
)
# we use the datetime to ensure the slug is unique
slug = f"it-generated-compute-specific-logits-demo-graph-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
# This method does everything:
# - Generates attribution graph
# - Transforms it for Neuronpedia compatibility
# - Uploads to Neuronpedia
# - Returns both the graph and the Neuronpedia metadata

# if you want to set custom metadata for Neuronpedia graph, you can do so here.
# in this demo, since the circuit-tracer generated graph does not currently have cantor-paired feature nodes, we
# need to avoid setting neuronpedia_source_set in the feature_details metadata
# temporary hack: https://bit.ly/non_cantor_edge_case
graph, local_graph_path, neuronpedia_metadata = ct_module.generate_graph(
    prompt=prompt,
    slug=slug,
    upload_to_np=False,  # Upload the produced graph to Neuronpedia if True, or just return the graph if False
)

print(f"Generated and saved graph locally to {local_graph_path}")

if neuronpedia_metadata and hasattr(neuronpedia_metadata, "url"):
    print("Graph uploaded successfully.")
    print(f"View your graph at: {neuronpedia_metadata.url}")

Phase 0: Precomputing activations and vectors


Precomputation completed in 1.47s


Found 5883 active features


Phase 1: Running forward pass


/mnt/cache/speediedan/.venvs/it_latest/lib/python3.13/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:139: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:408.)
  return input_acts @ self.W_skip.T


...

[INFO] interpretune.utils.logging: [NeuronpediaIntegration] Changes written to /tmp/20260810_010548_default/it-generated-compute-specific-logits-demo-graph-20260810-010609_qparam_changes_20260810_010617.log


[INFO] interpretune.utils.logging: [NeuronpediaIntegration] Transformed qParams for Neuronpedia


[INFO] interpretune.utils.logging: [NeuronpediaIntegration] Transformed graph saved to: /tmp/20260810_010548_default/graph_data/it-generated-compute-specific-logits-demo-graph-20260810-010609.json


[INFO] interpretune.utils.logging: Neuronpedia upload not requested. Set upload_to_np to `True` to automatically upload.


Generated and saved graph locally to /tmp/20260810_010548_default/graph_data/it-generated-compute-specific-logits-demo-graph-20260810-010609.json


In [9]:
# Alternative: Just transform an existing graph
# transformed_graph, np_metadata = ct_module.neuronpedia.transform_graph_for_np(
#     graph_path=local_graph_path,
#     # slug="it-generated-dallas-austin-demo-graph-20250725-121824-retransformed",
#     upload_to_np=False  # Set to True to upload the transformed graph
# )